<a href="https://colab.research.google.com/github/SABARINATH-KR/Audio-project---Music-Genre-Classification/blob/main/music_genre_clasification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import zipfile

# Download the GTZAN dataset
print('Downloading GTZAN dataset...')
path = kagglehub.dataset_download('andradaolteanu/gtzan-dataset-music-genre-classification')
print(f'Dataset downloaded to: {path}')

# Extraction setup
extraction_path = './gtzan_dataset'
if not os.path.exists(extraction_path):
    os.makedirs(extraction_path)

# The download from kagglehub usually provides a directory or a zip
# We check if we need to unzip or just link the path
if os.path.isdir(path):
    import shutil
    # If it's already a directory, we can use it or copy contents
    # For consistency with your next cell, we'll ensure the path exists
    print('Dataset directory identified.')
    # Update extraction_path to point to the kagglehub path for simplicity
    extraction_path = path
else:
    print(f'Unzipping {path}...')
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print('Dataset unzipped successfully!')

# Ensure the next cell uses the correct path
import sys
# Update the global genres_path variable if the cell below is already defined
genres_path = os.path.join(extraction_path, 'Data/genres_original')
print(f'Genres path set to: {genres_path}')

100%|██████████| 1.21G/1.21G [00:45<00:00, 28.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1
Dataset directory identified.
Genres path set to: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data/genres_original


In [2]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# The genres_path is already defined in the previous cell
print(f'Using dataset at: {genres_path}')

def extract_features(file_path):
    # Using a shorter duration for faster processing
    y, sr = librosa.load(file_path, duration=30)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    return np.mean(mfccs.T, axis=0)

features_list = []
labels = []

if os.path.exists(genres_path):
    print('Extracting features... this may take a few minutes.')
    for genre in os.listdir(genres_path):
        genre_dir = os.path.join(genres_path, genre)
        if os.path.isdir(genre_dir):
            for filename in os.listdir(genre_dir):
                # Updated to check for both .wav (current dataset) and .au extensions
                if filename.lower().endswith(('.au', '.wav')):
                    try:
                        features_list.append(extract_features(os.path.join(genre_dir, filename)))
                        labels.append(genre)
                    except Exception as e:
                        continue

    X = np.array(features_list)
    y = pd.Series(labels)

    if len(X) > 0:
        # Split and Scale
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # --- Model Training ---
        svc_model = SVC(kernel='rbf', random_state=42)
        print('Training SVC model...')
        svc_model.fit(X_train_scaled, y_train)

        y_pred = svc_model.predict(X_test_scaled)
        print(f'Model Accuracy: {accuracy_score(y_test, y_pred):.2f}')
        print('\nClassification Report:\n', classification_report(y_test, y_pred))
    else:
        print('No audio files found for extraction. Please check the file extensions in the dataset.')
else:
    print(f'Error: The path {genres_path} does not exist.')

Using dataset at: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data/genres_original
Extracting features... this may take a few minutes.


/tmp/ipykernel_794/3833951755.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=30)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Training SVC model...
Model Accuracy: 0.55

Classification Report:
               precision    recall  f1-score   support

       blues       0.67      0.50      0.57        20
   classical       0.75      0.90      0.82        20
     country       0.35      0.67      0.46        12
       disco       0.42      0.29      0.34        28
      hiphop       0.60      0.25      0.35        24
        jazz       0.75      0.57      0.65        21
       metal       0.68      1.00      0.81        15
         pop       0.54      0.88      0.67        17
      reggae       0.48      0.55      0.51        22
        rock       0.28      0.24      0.26        21

    accuracy                           0.55       200
   macro avg       0.55      0.58      0.54       200
weighted avg       0.55      0.55      0.53       200

